<a href="https://colab.research.google.com/github/sulucay01/multimodal-rs-segmentation/blob/dev/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Preprocessing

DI725 Term Project — Multimodal Fusion for Remote Sensing Land Cover Segmentation

This notebook produces two artifacts used by all downstream notebooks:
1. Class-index masks saved to `data/masks_class/`
2. `captions_with_splits.csv` with stratified train/val/test splits

Sections:
1. Setup
2. Mask conversion
3. Train/val/test split
4. Sanity check

## 1. Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Project paths
from pathlib import Path

PROJECT_DIR      = Path('/content/drive/MyDrive/DI725_Project')
DATA_DIR         = PROJECT_DIR / 'data'
IMAGES_DIR       = DATA_DIR / 'images'
MASKS_DIR        = DATA_DIR / 'masks'
MASKS_CLASS_DIR  = DATA_DIR / 'masks_class'
CAPTIONS_CSV     = DATA_DIR / 'captions.csv'
SPLITS_CSV       = DATA_DIR / 'captions_with_splits.csv'

MASKS_CLASS_DIR.mkdir(exist_ok=True)
assert DATA_DIR.exists(), f'Data dir not found: {DATA_DIR}'

In [ ]:
# Imports
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import train_test_split

SEED = 42

## 2. Mask conversion

The masks on disk are RGB images where each color encodes a land cover class. Models expect class indices (0–6) instead. This section converts all RGB masks once and saves them as uint8 PNGs to `data/masks_class/`, so downstream training reads class-index masks directly without per-batch conversion.

In [ ]:
# Class definitions from the project file
CLASS_NAMES = ['Tree', 'Shrub', 'Grass', 'Crop', 'Built-up', 'Barren', 'Water']

CLASS_RGB = {
    'Tree':     (0,   100, 0),
    'Shrub':    (255, 182, 193),
    'Grass':    (154, 205, 50),
    'Crop':     (255, 215, 0),
    'Built-up': (139, 69,  19),
    'Barren':   (211, 211, 211),
    'Water':    (0,   0,   255),
}
RGB_TO_IDX = {CLASS_RGB[name]: idx for idx, name in enumerate(CLASS_NAMES)}

df = pd.read_csv(CAPTIONS_CSV)
print(f'Rows: {len(df)}')

Rows: 10000


In [ ]:
def rgb_mask_to_class(mask_rgb):
    """Map HxWx3 RGB mask to HxW class indices (0..6) by exact color matching."""
    h, w, _ = mask_rgb.shape
    class_mask = np.zeros((h, w), dtype=np.uint8)
    for rgb, idx in RGB_TO_IDX.items():
        match = np.all(mask_rgb == np.array(rgb), axis=-1)
        class_mask[match] = idx
    return class_mask

In [ ]:
# Convert all RGB masks and save as class-index PNGs (uint8, single channel).
# The exists() check makes this safely re-runnable.
filenames = df['filename'].tolist()
unmatched_total = 0

for fname in tqdm(filenames, desc='Converting masks'):
    out_path = MASKS_CLASS_DIR / fname
    if out_path.exists():
        continue

    mask_rgb = np.array(Image.open(MASKS_DIR / fname).convert('RGB'))
    class_mask = rgb_mask_to_class(mask_rgb)

    # Track any pixels whose color did not match a class definition
    matched = np.zeros(mask_rgb.shape[:2], dtype=bool)
    for rgb in RGB_TO_IDX:
        matched |= np.all(mask_rgb == np.array(rgb), axis=-1)
    unmatched_total += (~matched).sum()

    Image.fromarray(class_mask, mode='L').save(out_path)

print(f'Done. Total unmatched pixels across all masks: {unmatched_total}')

Converting masks:   0%|          | 0/10000 [00:00<?, ?it/s]/tmp/ipykernel_515/3396810868.py:20: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(class_mask, mode='L').save(out_path)
Converting masks: 100%|██████████| 10000/10000 [47:38<00:00,  3.50it/s]

Done. Total unmatched pixels across all masks: 0


In [ ]:
# Verify by reconstructing class percentages from the saved masks for a few samples
sample_rows = df.sample(10, random_state=SEED)

print(f"{'Filename':<15} {'Class':<10} {'From mask':>10} {'From CSV':>10} {'Match?':>8}")
print('-' * 60)
for _, row in sample_rows.iterrows():
    cls_mask = np.array(Image.open(MASKS_CLASS_DIR / row['filename']))
    total = cls_mask.size
    unique, counts = np.unique(cls_mask, return_counts=True)
    for u, c in zip(unique, counts):
        mask_pct = c / total * 100
        csv_pct  = row[CLASS_NAMES[u]]
        match    = 'yes' if abs(mask_pct - csv_pct) < 2 else 'NO'
        print(f"{row['filename']:<15} {CLASS_NAMES[u]:<10} {mask_pct:>9.1f}% {csv_pct:>9.1f}% {match:>8}")

Filename        Class       From mask   From CSV   Match?
------------------------------------------------------------
267687.png      Tree            84.6%      85.0%      yes
267687.png      Shrub           13.4%      13.0%      yes
267687.png      Grass            2.0%       2.0%      yes
222448.png      Tree            18.6%      18.0%      yes
222448.png      Grass           80.7%      81.0%      yes
222448.png      Barren           0.7%       1.0%      yes
14328.png       Tree             1.7%       2.0%      yes
14328.png       Shrub            0.0%       0.0%      yes
14328.png       Grass           65.2%      65.0%      yes
14328.png       Crop             0.8%       1.0%      yes
14328.png       Built-up        22.1%      22.0%      yes
14328.png       Barren          10.1%      10.0%      yes
224249.png      Tree            60.9%      61.0%      yes
224249.png      Grass           39.1%      39.0%      yes
224249.png      Barren           0.0%       0.0%      yes
218094.png 

## 3. Train/val/test split

Stratified 70/15/15 split by the dominant class in each image (the class with the highest pixel percentage). Stratification ensures each split contains a proportional number of images for every dominant-class category, including rare ones like Built-up and Water.

In [ ]:
# Dominant class per image, used as the stratification key
df['dominant_class'] = df[CLASS_NAMES].idxmax(axis=1)

print('Dominant-class distribution across the full dataset:')
print(df['dominant_class'].value_counts().to_string())

Dominant-class distribution across the full dataset:
dominant_class
Grass       4703
Tree        3037
Crop        1803
Barren       193
Water        178
Built-up      59
Shrub         27


In [ ]:
# Two-step stratified split: first carve off 30% (val + test), then halve it.
train_df, temp_df = train_test_split(
    df, test_size=0.3, random_state=SEED, stratify=df['dominant_class']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['dominant_class']
)

train_df = train_df.copy(); train_df['split'] = 'train'
val_df   = val_df.copy();   val_df['split']   = 'val'
test_df  = test_df.copy();  test_df['split']  = 'test'

print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')

Train: 7000  |  Val: 1500  |  Test: 1500


In [ ]:
# Save merged split CSV
split_df = pd.concat([train_df, val_df, test_df]).sort_values('filename').reset_index(drop=True)
split_df.to_csv(SPLITS_CSV, index=False)
print(f'Saved: {SPLITS_CSV}')

Saved: /content/drive/MyDrive/DI725_Project/data/captions_with_splits.csv


## 4. Sanity check

Confirm the split is balanced across dominant classes, no filename appears in more than one split, and every row has a corresponding class-index mask.

In [ ]:
# Dominant-class counts per split
counts = pd.crosstab(split_df['dominant_class'], split_df['split'])
counts = counts[['train', 'val', 'test']].reindex(CLASS_NAMES).fillna(0).astype(int)
counts['total'] = counts.sum(axis=1)
counts

split,train,val,test,total
dominant_class,,,,
Tree,2126,455,456,3037
Shrub,19,4,4,27
Grass,3292,706,705,4703
Crop,1262,271,270,1803
Built-up,41,9,9,59
Barren,135,29,29,193
Water,125,26,27,178


In [ ]:
# No filename in multiple splits
overlap = split_df.duplicated(subset='filename').sum()
print(f'Filenames appearing in multiple splits: {overlap}')

# Every row has a corresponding class-index mask
missing = [f for f in split_df['filename'] if not (MASKS_CLASS_DIR / f).exists()]
print(f'Rows without a class-index mask: {len(missing)}')

Filenames appearing in multiple splits: 0
Rows without a class-index mask: 0


## Summary

`captions_with_splits.csv` contains every row from `captions.csv` plus a `split` column (`train`, `val`, or `test`) and a `dominant_class` column. Class-index masks for every image live in `data/masks_class/`. Downstream notebooks load this CSV, filter by `split`, and read masks from `data/masks_class/`.